# Modelo campeão: SVM linear + TF-IDF na contraparte

Escolhido em `exploracao.ipynb`. Este notebook trata só dele: o pipeline, o desempenho, o que ele aprendeu,
onde erra, e o treino do modelo final (`.pkl`).

O modelo vem de `ml/treinar.py` (`monta_modelo()`): é **o mesmo** que é salvo no `.pkl`, então o que se vê
aqui é exatamente o que a aplicação usa.

> Antes de fazer commit, limpe as saídas (**Clear All Outputs**): a seção 5 mostra pedaços de nomes reais de
> lojas e as outras saídas são derivadas dos extratos.

In [ ]:
import sys
from pathlib import Path

# permite importar o pacote ml com o notebook aberto em ml/ ou na raiz
RAIZ = Path.cwd().parent if Path.cwd().name == "ml" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score)
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_predict

from ml.treinar import monta_modelo

set_config(display="diagram")
pd.set_option("display.precision", 3)

## 1. Dados

In [ ]:
# funciona com o notebook aberto em ml/ ou na raiz do projeto
CSV = next(p for p in (Path("dados/transacoes_rotuladas.csv"), Path("ml/dados/transacoes_rotuladas.csv")) if p.exists())

# keep_default_na=False: contraparte vazia continua "" (sem contraparte), não vira NaN
df = pd.read_csv(CSV, keep_default_na=False, parse_dates=["data"])
y = df["categoria"]
X = df.drop(columns=["categoria"])  # exatamente o CSV do parser
grupos = df["id_usuario"]  # não é atributo do modelo: só define os grupos da validação por usuário

print(f"{len(df)} transações | {grupos.nunique()} usuários | {y.nunique()} classes")
y.value_counts().to_frame("transações").assign(**{"%": lambda t: (100 * t["transações"] / len(df)).round(1)})

## 2. O pipeline

Recebe o CSV do parser como está (as colunas `id_usuario` e `data` são ignoradas):

| Etapa | Colunas | O que faz |
|---|---|---|
| TF-IDF de caracteres | `contraparte` | nome da loja → pesos de pedaços de 3 a 5 letras |
| One-hot | `historico`, `meio`, `tipo`, `banco` | uma coluna 0/1 por valor (raros → "infrequent") |
| abs → log → min-max | `valor` | tira o sinal (está em `tipo`), reduz a cauda longa, escala para [0, 1] |
| SVM linear | tudo | um hiperplano por classe (um-contra-todos) |

In [ ]:
modelo = monta_modelo()
modelo  # clique nas caixas do diagrama para ver os parâmetros

## 3. Desempenho

Cada transação é prevista por um modelo que **não a viu no treino** (`cross_val_predict`, 5 folds),
nas duas validações.

In [ ]:
previsoes = {
    "aleatória": cross_val_predict(modelo, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42), n_jobs=-1),
    "por usuário": cross_val_predict(modelo, X, y, cv=GroupKFold(5), groups=grupos, n_jobs=-1),
}
pd.DataFrame({nome: {"acurácia": accuracy_score(y, p), "F1 macro": f1_score(y, p, average="macro")}
              for nome, p in previsoes.items()}).T

In [ ]:
print("Validação por usuário — por classe:\n")
print(classification_report(y, previsoes["por usuário"], digits=3))

In [ ]:
classes = sorted(y.unique())
fig, eixos = plt.subplots(1, 2, figsize=(18, 7.5))
for eixo, (nome, p) in zip(eixos, previsoes.items()):
    ConfusionMatrixDisplay.from_predictions(y, p, labels=classes, normalize="true", values_format=".0%",
                                            cmap="Blues", colorbar=False, xticks_rotation=45, ax=eixo)
    eixo.set_title(f"Validação {nome}: acurácia {accuracy_score(y, p):.1%} | F1 macro {f1_score(y, p, average='macro'):.1%}")
    eixo.set_xlabel("previsto")
    eixo.set_ylabel("real")
plt.tight_layout()
plt.show()

## 4. Desempenho por usuário

Na validação por usuário, cada usuário é previsto por um modelo que nunca viu nenhuma transação dele.
Usuários com banco único nos dados (o histórico do banco é novo para o modelo) tendem a ir pior.

In [ ]:
p = pd.Series(previsoes["por usuário"], index=y.index)
por_usuario = pd.DataFrame({
    "transações": grupos.value_counts(),
    "banco(s)": df.groupby("id_usuario")["banco"].agg(lambda b: ", ".join(sorted(set(b)))),
    "acurácia": y.groupby(grupos).apply(lambda s: accuracy_score(s, p[s.index])),
    "F1 macro": y.groupby(grupos).apply(lambda s: f1_score(s, p[s.index], average="macro")),
}).sort_values("acurácia")

ax = por_usuario[["acurácia", "F1 macro"]].plot.barh(figsize=(9, 4.5), xlim=(0, 1), title="Validação por usuário")
ax.axvline(accuracy_score(y, p), color="gray", linestyle="--", linewidth=1)
plt.tight_layout()
plt.show()
por_usuario

## 5. O que o modelo aprendeu

O SVM linear dá um peso a cada atributo, por classe: peso alto = atributo que empurra para aquela classe.
Abaixo, os atributos de maior peso de cada classe, treinando com todos os dados.
(`contraparte__xyz` = pedaço de nome de loja; `cat__meio_pix` = meio igual a pix; etc.)

In [ ]:
final = monta_modelo().fit(X, y)
nomes = final.named_steps["prep"].get_feature_names_out()
pesos = final.named_steps["clf"].coef_  # uma linha por classe

N = 8
fig, eixos = plt.subplots(2, 4, figsize=(20, 8))
for eixo, classe, w in zip(eixos.ravel(), final.classes_, pesos):
    topo = np.argsort(w)[-N:]
    eixo.barh([nomes[i].replace("contraparte__", "loja: ").replace("cat__", "").replace("valor__", "")
               for i in topo], w[topo], color="tab:blue")
    eixo.set_title(classe)
plt.suptitle(f"Os {N} atributos de maior peso por classe", y=1.0)
plt.tight_layout()
plt.show()

## 6. Onde ele erra

Pares (real → previsto) mais frequentes na validação por usuário. O padrão esperado: quando não reconhece
a loja de um usuário novo, o modelo tende a `compras`.

In [ ]:
erros = df.assign(previsto=previsoes["por usuário"]).query("categoria != previsto")
print(f"{len(erros)} erros de {len(df)} ({len(erros) / len(df):.1%})")
erros.groupby(["categoria", "previsto", "meio"]).size().sort_values(ascending=False).head(12).to_frame("erros")

## 7. Modelo final (`.pkl`)

Treina com **todas** as transações rotuladas e salva em `ml/modelos/svm_tfidf.pkl` (fora do git).
É o mesmo que `python -m ml.treinar`.

In [ ]:
import joblib

from ml.treinar import treina

pkl = RAIZ / "ml" / "modelos" / "svm_tfidf.pkl"
modelo_final, n = treina(CSV)
pkl.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(modelo_final, pkl)
print(f"treinado com {n} transações e salvo em {pkl}")

# uso: recebe o CSV do parser (sem categoria) e devolve a categoria
carregado = joblib.load(pkl)
pd.Series(carregado.predict(X.head(10)), name="categoria_prevista").to_frame()